### Hi team.. this notebook is a bit of a mess as I was trying to figure out the cleanest workflow lol... but I put included it here just to have some documentation for how I pulled 2016 data, but the actual clean, reproducible code for 2017-2019 is in `notebooks/cropscape-data-cleaning.ipynb`. Additionally, the relative file paths in this notebook have not been updated since I initially pulled the 2016 data from CropScape but I didn't want to re-run with my new reproducible code to be kind to the server <3

### Imports

In [27]:
import pandas as pd
import requests
import re
from io import StringIO
from time import sleep

### make the county FIPS list from your existing counties dataframe
*add link to download here* + CODE DOES NOT INCLUDE READING IN COUNTIES FILE YET

In [28]:
county_fips_list = counties["county_fips"].astype(str).tolist()

len(county_fips_list), county_fips_list[:10]

(3220,
 ['01001',
  '01003',
  '01005',
  '01007',
  '01009',
  '01011',
  '01013',
  '01015',
  '01017',
  '01019'])

### helper function to fetch one county’s CDL stats

In [29]:
BASE_URL = "https://nassgeodata.gmu.edu/axis2/services/CDLService/GetCDLStat"

def fetch_one_county_cdl(year, fips, timeout=120):
    params = {
        "year": year,
        "fips": str(fips).zfill(5),
        "format": "csv"
    }

    # Step 1: hit the API endpoint
    r = requests.get(BASE_URL, params=params, timeout=timeout)
    r.raise_for_status()

    # Step 2: extract the returnURL from the XML response
    match = re.search(r"<returnURL>(.*?)</returnURL>", r.text)
    if not match:
        raise ValueError(f"Could not find returnURL for FIPS {fips}")

    csv_url = match.group(1)

    # Step 3: download the actual CSV
    csv_response = requests.get(csv_url, timeout=timeout)
    csv_response.raise_for_status()

    # Step 4: read into pandas
    df = pd.read_csv(StringIO(csv_response.text))
    df["county_fips"] = str(fips).zfill(5)
    df["year"] = year

    return df

### quick test on one county

In [30]:
test_df = fetch_one_county_cdl(2016, "01001")
print(test_df.shape)
test_df.head()

(37, 6)


,Value,Category,Count,Acreage,county_fips,year
0,1,Corn,7220,1605.7,01001,2016
1,2,Cotton,40741,9060.6,01001,2016
2,4,Sorghum,334,74.3,01001,2016
3,5,Soybeans,7888,1754.2,01001,2016
4,10,Peanuts,384,85.4,01001,2016


### loop through all counties for 2016, with retry tracking

In [31]:
year = 2016

frames = []
failed_counties = []

for i, fips in enumerate(county_fips_list, start=1):
    try:
        df = fetch_one_county_cdl(year, fips)
        frames.append(df)

        if i % 50 == 0:
            print(f"Completed {i} counties")

    except Exception as e:
        failed_counties.append({
            "county_fips": fips,
            "error": str(e)
        })
        print(f"FAILED: {fips} -> {e}")

    sleep(0.2)  # be gentle with the server

Completed 50 counties
FAILED: 02013 -> 500 Server Error: Internal Server Error for url: https://nassgeodata.gmu.edu/axis2/services/CDLService/GetCDLStat?year=2016&fips=02013&format=csv
FAILED: 02016 -> 500 Server Error: Internal Server Error for url: https://nassgeodata.gmu.edu/axis2/services/CDLService/GetCDLStat?year=2016&fips=02016&format=csv
FAILED: 02020 -> 500 Server Error: Internal Server Error for url: https://nassgeodata.gmu.edu/axis2/services/CDLService/GetCDLStat?year=2016&fips=02020&format=csv
FAILED: 02050 -> 500 Server Error: Internal Server Error for url: https://nassgeodata.gmu.edu/axis2/services/CDLService/GetCDLStat?year=2016&fips=02050&format=csv
FAILED: 02060 -> 500 Server Error: Internal Server Error for url: https://nassgeodata.gmu.edu/axis2/services/CDLService/GetCDLStat?year=2016&fips=02060&format=csv
FAILED: 02068 -> 500 Server Error: Internal Server Error for url: https://nassgeodata.gmu.edu/axis2/services/CDLService/GetCDLStat?year=2016&fips=02068&format=csv


KeyboardInterrupt: 

### combine and save what worked
Interrupted kernel  while running code chunk above due to large size and fear of losing progress. Intermittently saved progress after interrupting below and will re-write loop code  further below to include intermittent saving.

In [32]:
progress_df = pd.concat(frames, ignore_index=True)
progress_df.to_csv("raw-files/cdl_2016_progress_backup.csv", index=False)

failed_df = pd.DataFrame(failed_counties)
failed_df.to_csv("raw-files/cdl_failed_progress_2016.csv", index=False)

print(progress_df.shape)
print(failed_df.shape)

(36182, 6)
(34, 2)


### reload the saved progress

In [34]:
import pandas as pd

progress_df = pd.read_csv("raw-files/cdl_2016_progress_backup.csv", dtype={"county_fips": str})
failed_df = pd.read_csv("raw-files/cdl_failed_progress_2016.csv", dtype={"county_fips": str})

progress_df["county_fips"] = progress_df["county_fips"].str.zfill(5)
failed_df["county_fips"] = failed_df["county_fips"].str.zfill(5)

print(progress_df.shape)
print(failed_df.shape)
print(progress_df["county_fips"].nunique())

(36182, 6)
(34, 2)
1175


### compute remaining counties

In [35]:
done_fips = set(progress_df["county_fips"].unique())

remaining_fips = [
    f for f in county_fips_list
    if str(f).zfill(5) not in done_fips
]

print("Completed counties:", len(done_fips))
print("Remaining counties:", len(remaining_fips))

Completed counties: 1175
Remaining counties: 2045


### run the resume loop with checkpoints

In [36]:
import os

year = 2016

append_file = "raw-files/cdl_2016_progress_append.csv"
failed_append_file = "raw-files/cdl_failed_progress_append.csv"

new_failed = []

for i, fips in enumerate(remaining_fips, start=1):

    try:
        df = fetch_one_county_cdl(year, fips)

        df.to_csv(
            append_file,
            mode="a",
            header=not os.path.exists(append_file),
            index=False
        )

    except Exception as e:
        new_failed.append({
            "county_fips": fips,
            "error": str(e)
        })
        print(f"FAILED: {fips}")

    if i % 50 == 0:
        print(f"Downloaded {i} remaining counties")

        pd.DataFrame(new_failed).to_csv(
            failed_append_file,
            index=False
        )

    sleep(0.2)

FAILED: 02013
FAILED: 02016
FAILED: 02020
FAILED: 02050
FAILED: 02060
FAILED: 02068
FAILED: 02070
FAILED: 02090
FAILED: 02100
FAILED: 02105
FAILED: 02110
FAILED: 02122
FAILED: 02130
FAILED: 02150
FAILED: 02158
FAILED: 02164
FAILED: 02170
FAILED: 02180
FAILED: 02185
FAILED: 02188
FAILED: 02195
FAILED: 02198
FAILED: 02220
FAILED: 02230
FAILED: 02240
FAILED: 02261
FAILED: 02275
FAILED: 02282
FAILED: 02290
FAILED: 15001
FAILED: 15003
FAILED: 15005
FAILED: 15007
FAILED: 15009
Downloaded 50 remaining counties
Downloaded 100 remaining counties
Downloaded 150 remaining counties
Downloaded 200 remaining counties


KeyboardInterrupt: 

In [37]:
exclude_state_fips = {"02", "15", "60", "66", "69", "72", "78"}

done_fips = set(progress_df["county_fips"].astype(str).str.zfill(5).unique())

remaining_fips_conus = [
    str(f).zfill(5)
    for f in county_fips_list
    if str(f).zfill(5) not in done_fips
    and str(f).zfill(5)[:2] not in exclude_state_fips
]

print("Remaining CONUS counties:", len(remaining_fips_conus))
print(remaining_fips_conus[:20])

Remaining CONUS counties: 1933
['24035', '24037', '24039', '24041', '24043', '24045', '24047', '24510', '25001', '25003', '25005', '25007', '25009', '25011', '25013', '25015', '25017', '25019', '25021', '25023']


### re-run with CONUS only

In [38]:
import os

year = 2016

append_file = "raw-files/cdl_2016_progress_append.csv"
failed_append_file = "raw-files/cdl_failed_progress_append.csv"

new_failed = []

for i, fips in enumerate(remaining_fips_conus, start=1):
    try:
        df = fetch_one_county_cdl(year, fips)

        df.to_csv(
            append_file,
            mode="a",
            header=not os.path.exists(append_file),
            index=False
        )

        if i % 50 == 0:
            print(f"Downloaded {i} remaining CONUS counties")

    except Exception as e:
        new_failed.append({
            "county_fips": str(fips).zfill(5),
            "error": str(e)
        })
        print(f"FAILED: {fips} -> {e}")

    if i % 50 == 0:
        pd.DataFrame(new_failed).to_csv(
            failed_append_file,
            index=False
        )

    sleep(0.2)

Downloaded 50 remaining CONUS counties
Downloaded 100 remaining CONUS counties
Downloaded 150 remaining CONUS counties
Downloaded 200 remaining CONUS counties
Downloaded 250 remaining CONUS counties
Downloaded 300 remaining CONUS counties
Downloaded 350 remaining CONUS counties
Downloaded 400 remaining CONUS counties
Downloaded 450 remaining CONUS counties
Downloaded 500 remaining CONUS counties
Downloaded 550 remaining CONUS counties
Downloaded 600 remaining CONUS counties
Downloaded 650 remaining CONUS counties
Downloaded 700 remaining CONUS counties
Downloaded 750 remaining CONUS counties
Downloaded 800 remaining CONUS counties
Downloaded 850 remaining CONUS counties
Downloaded 900 remaining CONUS counties
Downloaded 950 remaining CONUS counties
Downloaded 1000 remaining CONUS counties
Downloaded 1050 remaining CONUS counties
Downloaded 1100 remaining CONUS counties
Downloaded 1150 remaining CONUS counties
Downloaded 1200 remaining CONUS counties
Downloaded 1250 remaining CONUS coun

### combine old + new results

In [39]:
original = pd.read_csv("raw-files/cdl_2016_progress_backup.csv", dtype={"county_fips": str})
new = pd.read_csv("raw-files/cdl_2016_progress_append.csv", dtype={"county_fips": str})

cdl_raw_2016 = pd.concat([original, new], ignore_index=True)

cdl_raw_2016.to_csv("raw-files/cdl_raw_2016_complete.csv", index=False)

print(cdl_raw_2016.shape)
print("Unique counties:", cdl_raw_2016["county_fips"].nunique())

(108080, 6)
Unique counties: 3106


### append successful retries and resave

### inspect the columns before cleaning